# Agentic RAG Using Langraph

In [16]:
import os
import time
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from huggingface_hub import get_collection
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google import genai
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader

In [17]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2591.97it/s]


In [18]:
parser = StrOutputParser()

In [19]:
#all api keys

#QDRANT
QDRANT_API_KEY = os.getenv("QDRANTAPIKEY")
QDRANT_ENDPOINT = os.getenv("QDRANTENDPOINT")

#gemini model apikey
geminiapikey = os.getenv("GEMINIAPIKEY")

#
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite",
#     google_api_key=geminiapikey,
#     temperature=0
# )


groq_api_key = os.getenv("GROQAPIKEY")  # make sure this matches your .env variable name

llm = ChatGroq(
    model="qwen/qwen3-32b",
    api_key=groq_api_key,
    temperature=0
)

In [20]:
start = time.time()
try:
    test = llm.invoke("Say hello in one word.")
    print("SUCCESS:", test)
except Exception as e:
    print("FAILED:", e)
print(f"Took {time.time() - start:.1f}s")

SUCCESS: content='<think>\nOkay, the user wants me to say hello in one word. Let me think about the possible options. The most straightforward is "Hello" itself, but maybe they want a different approach. Words like "Hi" or "Hey" are shorter, but still one word. Alternatively, maybe a greeting in another language? Like "Bonjour" or "Ciao." But the user didn\'t specify a language, so sticking to English is safer. Let me check if there\'s any other single-word greetings. "Hey" is one word, "Hi" is another. "Hello" is the standard. Since the user asked for one word, "Hello" fits perfectly. I don\'t think there\'s any ambiguity here. Just make sure to capitalize it correctly. Yeah, that should be it.\n</think>\n\nHello' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 14, 'total_tokens': 180, 'completion_time': 0.353828872, 'completion_tokens_details': None, 'prompt_time': 0.000333814, 'prompt_tokens_details': None, 'queue_time': 0.006365388

In [21]:
file_path = "who.pdf"
loader = PyPDFLoader(file_path)

document = loader.load()

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)


In [22]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=130,
    separators=[
        "\n\n",   # paragraphs (highest priority)
        "\n",     # lines
        ". ",     # sentences
        " ",      # words
        ""        # fallback
    ]
)

chunks = splitter.split_documents(document)

In [23]:
client = QdrantClient(
    url=QDRANT_ENDPOINT,
    api_key=QDRANT_API_KEY
)

collections = client.get_collections().collections

collection_exists = any(
    collection.name == "agentic_rag"
    for collection in collections
)

# =========================
# 7. Create Collection Only If Not Exists
# =========================

if not collection_exists:

    vectorstore = QdrantVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings,
        url=QDRANT_ENDPOINT,
        api_key=QDRANT_API_KEY,
        collection_name="agentic_rag",
    )

    print("Collection created and documents stored.")

else:

    vectorstore = QdrantVectorStore.from_existing_collection(
        embedding=embeddings,
        url=QDRANT_ENDPOINT,
        api_key=QDRANT_API_KEY,
        collection_name="agentic_rag",
    )

    print("Collection already exists. Using existing collection.")

Collection already exists. Using existing collection.


In [24]:
from typing import TypedDict, List, Optional


In [25]:

class AgentState(TypedDict):
    """State schema for our Agentic RAG system.

    This is the shared clipboard that every node can read and write to.
    """
    question: str
    documents: List[str]
    answer: str
    iterations: int
    search_type: str   #vectorstore or web search


In [26]:
def retrieve(state: AgentState)->dict:
    """Retrieve the relevant documents from the database"""

    print(f"  [RETRIEVE] Searching for: {state['question'][:50]}...")

    docs = vectorstore.invoke(state["question"])

    docs_text = [doc.page_content for doc in docs]

    print(f"  [RETRIEVE] Found {len(docs_text)} documents")

    return {
        "documents": docs_text,
        "search_type": "vectorstore",
    }



In [27]:
grade_prompt = ChatPromptTemplate.from_template(
    """You are a document relevance grader.
Determine if the following document is relevant to the question.
Reply with ONLY 'yes' or 'no'.

Document: {document}

Question: {question}

Is this document relevant? (yes/no):"""
)

In [28]:
def grade_documents(state: AgentState)->dict:

    relevant_docs = []

    grade_chain = grade_prompt | llm | StrOutputParser()

    for doc in state["documents"]:
        result = grade_chain.invoke({
            "documents": doc,
            "question": state["question"],
        })
        if "yes" in result.strip().lower():
            relevant_docs.append(doc)

    print(f"  [GRADE] {len(relevant_docs)}/{len(state['documents'])} documents are relevant")

    return {"documents": relevant_docs}



In [29]:
generate_prompt = ChatPromptTemplate.from_template(
    """
Answer the question based ONLY on the following context.
If the context does not contain the answer, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:"""
)

In [30]:
def generate(state: AgentState)->dict:
    context = "/n/n".join(state["documents"])  #    # Join all relevant documents into one context string

    generate_chain = generate_prompt | llm | StrOutputParser()

    answer = generate_chain.invoke({
        "context": context,
        "question": state["question"],
    })

    print(f"  [GENERATE] Answer created ({len(answer)} characters)")

    return {
        "answer": answer,
        "iterations": state.get("iterations", 0) + 1,
    }

In [31]:
evaluate_prompt = ChatPromptTemplate.from_template(
    """You are an answer quality evaluator.
Determine if the following answer adequately addresses the question.
Consider: Is it factual? Is it complete? Does it address the question?

Question: {question}
Answer: {answer}

Reply with ONLY 'good' or 'not_good':"""
)

In [32]:
def evaluate_answer(state: AgentState) -> dict:
    """Evaluate if the generated answer is good enough."""
    print(f"  [EVALUATE] Checking answer quality (attempt {state.get('iteration_count', 1)})...")

    eval_chain = evaluate_prompt | llm | StrOutputParser()

    result = eval_chain.invoke({
        "question": state["question"],
        "answer": state["answer"],
    })

    quality = result.strip().lower()
    print(f"  [EVALUATE] Quality: {quality}")

    # Store the evaluation result in the answer field
    # (the routing function will check this)
    return {"answer": state["answer"]}

In [33]:
MAX_ITERATIONS = 3  # Maximum retry attempts to prevent infinite loops


def route_after_grading(state: AgentState) -> str:
    """Decide what to do after grading documents.

    Returns:
        'generate' if we have relevant documents
        'web_search' if no relevant documents found
    """
    if state["documents"] and len(state["documents"]) > 0:
        print(f"  [ROUTE] Relevant docs found -> Generate answer")
        return "generate"
    else:
        print(f"  [ROUTE] No relevant docs -> Web search fallback")
        return "web_search"


def route_after_evaluation(state: AgentState) -> str:
    """Decide what to do after evaluating the answer.

    Returns:
        'end' if answer is good or max iterations reached
        'retrieve' if answer needs improvement (retry)
    """
    iteration = state.get("iteration_count", 1)

    # Safety: stop after MAX_ITERATIONS to prevent infinite loops
    if iteration >= MAX_ITERATIONS:
        print(f"  [ROUTE] Max iterations ({MAX_ITERATIONS}) reached -> End")
        return "end"

    # For this demo, we always accept after evaluation
    # In production, you would check the evaluation result
    print(f"  [ROUTE] Answer accepted -> End")
    return "end"


print("[SUCCESS] Routing functions defined:")
print(f"  route_after_grading()    - Relevant docs? Generate : Web Search")
print(f"  route_after_evaluation() - Good answer? End : Retry (max {MAX_ITERATIONS})")

[SUCCESS] Routing functions defined:
  route_after_grading()    - Relevant docs? Generate : Web Search
  route_after_evaluation() - Good answer? End : Retry (max 3)
